In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/cnndataset'
video_data_path = os.path.join(base_path, 'Boxing_clip')
image_data_path = os.path.join(base_path, 'punch_imgs')

print("Checking paths...")
if os.path.exists(video_data_path):
    print(f"✅ Video Path Found: {video_data_path}")
    print("Classes:", os.listdir(video_data_path))
else:
    print(f"❌ Video Path Not Found: {video_data_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking paths...
✅ Video Path Found: /content/drive/MyDrive/cnndataset/Boxing_clip
Classes: ['1', '2', '3']


In [ ]:
%%writefile /content/drive/MyDrive/cnndataset/dataset.py
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
from PIL import Image

class BoxingVideoDataset(Dataset):
    def __init__(self, root_dir, seq_len=16, transform=None):
        """
        root_dir: مسار مجلد Boxing_clip الذي يحتوي على 1, 2, 3
        """
        self.root_dir = root_dir
        self.seq_len = seq_len
        self.transform = transform
        self.samples = []


        self.class_map = {'1': 0, '2': 1, '3': 2}


        for folder_name in ['1', '2', '3']:
            folder_path = os.path.join(root_dir, folder_name)
            if not os.path.exists(folder_path):
                continue

            label = self.class_map[folder_name]
            for file_name in os.listdir(folder_path):
                if file_name.lower().endswith(('.mov', '.mp4', '.avi')):
                    self.samples.append((os.path.join(folder_path, file_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frames = self._load_video(path)


        video_tensor = torch.stack(frames).permute(1, 0, 2, 3)

        return video_tensor, label

    def _load_video(self, path):
        """
        Data Cleaning Step:
        - قراءة الفيديو
        - توحيد عدد الفريمات (Sampling)
        - تغيير الحجم (Resizing)
        """
        cap = cv2.VideoCapture(path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


        if total_frames > self.seq_len:
            indices = np.linspace(0, total_frames-1, self.seq_len).astype(int)
        else:
            indices = np.arange(total_frames)

        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret: break

            if frame_idx in indices:

                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                frame = cv2.resize(frame, (224, 224))


                tensor_frame = torch.tensor(frame).float() / 255.0
                tensor_frame = tensor_frame.permute(2, 0, 1) # (C, H, W)
                frames.append(tensor_frame)

            frame_idx += 1
            if len(frames) == self.seq_len: break

        cap.release()



        while len(frames) < self.seq_len:
            last_frame = frames[-1] if len(frames) > 0 else torch.zeros(3, 224, 224)
            frames.append(last_frame)

        return frames


# 2. HitNet Dataset (Image Classification with Attention)
class BoxingHitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        root_dir: مسار مجلد punch_imgs
        """
        self.root_dir = root_dir
        self.transform = transform
        self.data = []

        # Data Cleaning: استخراج الـ Label من اسم الملف
        # punch_miss46.png -> Label 0
        # punch_land118.jpg -> Label 1
        valid_extensions = ('.jpg', '.jpeg', '.png')
        for img_name in os.listdir(root_dir):
            if not img_name.lower().endswith(valid_extensions):
                continue

            img_path = os.path.join(root_dir, img_name)

            if "miss" in img_name.lower():
                label = 0 # Miss
            elif "land" in img_name.lower():
                label = 1 # Land
            else:
                continue # تجاهل الملفات غير الواضحة

            self.data.append((img_path, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        # استخدام PIL للقراءة
        image = Image.open(img_path).convert("RGB")
        image = image.resize((224, 224))

        # التحويل لـ Tensor
        image = torch.tensor(np.array(image)).float() / 255.0
        image = image.permute(2, 0, 1) # (C, H, W)

        if self.transform:
             image = self.transform(image)

        return image, label

Writing /content/drive/MyDrive/cnndataset/dataset.py


In [ ]:
import sys
import os


my_project_path = '/content/drive/MyDrive/cnndataset'

if my_project_path not in sys.path:
    sys.path.append(my_project_path)

# 2. دلوقتي نقدر نعمل Import للكلاسات
from dataset import BoxingVideoDataset, BoxingHitDataset

print("✅ تم استدعاء الكلاسات بنجاح! جاهز للاختبار.")

✅ تم استدعاء الكلاسات بنجاح! جاهز للاختبار.


In [ ]:
import torch
from torch.utils.data import DataLoader
import os

# 1. إعداد المسارات
# ---------------------------------------------------------
# تأكد أنك قمت بعمل Mount للـ Drive في بداية النوت بوك
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/cnndataset'
video_root = os.path.join(base_path, 'Boxing_clip')
image_root = os.path.join(base_path, 'punch_imgs')

print(f"Checking paths...")
if os.path.exists(video_root):
    print(f"✅ Video path found: {video_root}")
else:
    print(f"❌ Video path NOT found: {video_root}")

# ---------------------------------------------------------
# 2. اختبار داتا الفيديو (ActionNet)
# ---------------------------------------------------------
print("\n--- Testing ActionNet Dataset (Video) ---")
try:
    # هنا يستدعي الكلاس مباشرة لأنه موجود في الخلية السابقة
    video_ds = BoxingVideoDataset(video_root, seq_len=16)

    if len(video_ds) > 0:
        video_loader = DataLoader(video_ds, batch_size=2, shuffle=True)
        videos, labels = next(iter(video_loader))

        print(f"✅ Success! Found {len(video_ds)} videos.")
        print(f"   Shape of batch: {videos.shape}")
        print("   (Batch_Size, Channels, Frames, Height, Width)")
        print(f"   Labels: {labels}")
    else:
        print("⚠️ Warning: The dataset is empty (0 videos found). Check subfolders 1, 2, 3.")

except Exception as e:
    print(f"❌ Error in Video Dataset: {e}")

# ---------------------------------------------------------
# 3. اختبار داتا الصور (HitNet)
# ---------------------------------------------------------
print("\n--- Testing HitNet Dataset (Images) ---")
try:
    image_ds = BoxingHitDataset(image_root)

    if len(image_ds) > 0:
        image_loader = DataLoader(image_ds, batch_size=4, shuffle=True)
        images, labels = next(iter(image_loader))

        print(f"✅ Success! Found {len(image_ds)} images.")
        print(f"   Shape of batch: {images.shape}")
        print("   (Batch_Size, Channels, Height, Width)")
        print(f"   Labels: {labels}")
    else:
        print("⚠️ Warning: The dataset is empty (0 images found). Check subfolders Land, Miss.")

except Exception as e:
    print(f"❌ Error in Image Dataset: {e}")

Checking paths...
✅ Video path found: /content/drive/MyDrive/cnndataset/Boxing_clip

--- Testing ActionNet Dataset (Video) ---
✅ Success! Found 246 videos.
   Shape of batch: torch.Size([2, 3, 16, 224, 224])
   (Batch_Size, Channels, Frames, Height, Width)
   Labels: tensor([0, 1])

--- Testing HitNet Dataset (Images) ---
✅ Success! Found 191 images.
   Shape of batch: torch.Size([4, 3, 224, 224])
   (Batch_Size, Channels, Height, Width)
   Labels: tensor([1, 1, 1, 1])
